In [27]:
import os
import time
import pandas as pd
import chromadb



from minsearch import Index
from sentence_transformers import CrossEncoder
from chromadb.utils import embedding_functions
from dotenv import load_dotenv
from groq import Groq

## Getting Data

In [16]:
DATA_PATH = "data_kaggle/google_books_1299.csv"

df = pd.read_csv(DATA_PATH)

df = df.fillna("")

df = df[df["description"].str.strip() != ""]

print(f"Total books: {len(df)}")

Total books: 1296


In [18]:
books = []

for i, row in df.iterrows():

    genre = str(row["generes"])

    # Corrige HTML presente no dataset
    genre = (
        genre
        .replace("&amp;", "&")
        .replace(" ,", ",")
    )

    books.append({
        "id": str(i),
        "title": str(row["title"]),
        "author": str(row["author"]),
        "genre": genre,
        "description": str(row["description"]),
        "published_date": str(row["published_date"])
    })


print(f"Books: {len(books)}")

Books: 1296


## Keyword Index

In [19]:
keyword_index = Index(
    text_fields=[
        "title",
        "author",
        "genre",
        "description"
    ]
)

keyword_index.fit(books)

## Keyword Search

In [20]:
def keyword_search(query, k=10):

    return keyword_index.search(
        query=query,
        boost_dict={
            "title": 1.0,
            "genre": 2.0,
            "description": 4.0,
            "author": 1.0
        },
        num_results=k
    )

## Vector Database

In [21]:
embedding_function = (
    embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
)

chroma_client = chromadb.PersistentClient(
    path="./vectordb_books"
)

try:
    chroma_client.delete_collection("books")
except Exception:
    pass

In [22]:
collection = chroma_client.get_or_create_collection(
    name="books",
    embedding_function=embedding_function
)

## Documents

In [23]:
ids = []
documents = []
metadatas = []


for book in books:

    text = f"""
Title: {book['title']}

Author: {book['author']}

Genre: {book['genre']}

Description:
{book['description']}
"""

    ids.append(book["id"])
    documents.append(text)

    metadatas.append({
        "title": book["title"],
        "author": book["author"],
        "genre": book["genre"]
    })


collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas
)


print(f"Vectors stored: {collection.count()}")

Vectors stored: 1296


## Vector Search

In [25]:
def vector_search(query, k=10):

    result = collection.query(
        query_texts=[query],
        n_results=k
    )

    results = []

    for i in range(len(result["ids"][0])):

        results.append({
            "id": result["ids"][0][i],
            "title": result["metadatas"][0][i]["title"],
            "author": result["metadatas"][0][i]["author"],
            "genre": result["metadatas"][0][i]["genre"],
            "description": result["documents"][0][i]
        })

    return results

## Hibryd Search

In [26]:
def hybrid_search(query, k=10):

    keyword_results = keyword_search(query, k)
    vector_results = vector_search(query, k)

    scores = {}

    # Reciprocal Rank Fusion
    for rank, book in enumerate(keyword_results, 1):

        book_id = book["id"]

        scores[book_id] = (
            scores.get(book_id, 0)
            + 1 / (60 + rank)
        )


    for rank, book in enumerate(vector_results, 1):

        book_id = book["id"]

        scores[book_id] = (
            scores.get(book_id, 0)
            + 1 / (60 + rank)
        )


    book_map = {
        book["id"]: book
        for book in books
    }


    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )


    return [
        book_map[book_id]
        for book_id in ranked_ids[:k]
    ]

## Reranker

In [ ]:
reranker = CrossEncoder(
    "cross-encoder/ms-marco-MiniLM-L-6-v2"
)


def rerank(query, results):

    pairs = []

    for book in results:

        text = f"""
Title: {book['title']}
Genre: {book['genre']}
Description: {book['description']}
"""

        pairs.append(
            (query, text)
        )


    scores = reranker.predict(pairs)


    ranked = sorted(
        zip(results, scores),
        key=lambda x: x[1],
        reverse=True
    )


    return [
        book
        for book, score in ranked
    ]



## Groq

In [28]:
load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

if not api_key:

    raise RuntimeError(
        "GROQ_API_KEY not found."
    )


groq_client = Groq(
    api_key=api_key
)


## RAG

In [ ]:
def ask_book_rag(query):

    start = time.time()

    candidates = hybrid_search(
        query,
        k=10
    )

    ranked_books = rerank(
        query,
        candidates
    )

    top_books = ranked_books[:5]

    context = "\n\n".join(

        f"""
Title: {book['title']}
Author: {book['author']}
Genre: {book['genre']}
Description: {book['description']}
"""

        for book in top_books
    )


    prompt = f"""
You are a book recommendation assistant.

User request:

{query}

Recommend books from the provided context
that best match the user's request.

For each recommendation:
- give the title
- give the author
- explain briefly why it matches

Use ONLY information from the context.

Do not invent books.
Do not invent information.

If the context contains books that are reasonably
related to the request, recommend them.

Do not recomend the exact book asked in the query.

Always give an idication even if does note have explicitly the exact words in query

Context:

{context}
"""
    available_models = {
        model.id
        for model in groq_client.models.list().data
    }


    preferred_models = [
        "llama-4-scout-17b-16e-instruct",
        "llama-3.3-70b-versatile",
        "openai/gpt-oss-20b",
        "llama-3.1-8b-instant"
    ]


    model = next(
        (
            m
            for m in preferred_models
            if m in available_models
        ),
        None
    )

    if model is None:

        raise RuntimeError(
            "No compatible Groq model found."
        )

    response = groq_client.chat.completions.create(

        model=model,

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        temperature=0
    )

    answer = response.choices[0].message.content

    latency = time.time() - start

    return {
        "query": query,
        "answer": answer,
        "books": top_books,
        "model": model,
        "latency": latency
    }   

In [30]:
query = "recommend fantasy books about mystery and alchemy"

result = ask_book_rag(query)

print(result["answer"])

**1. Kings and Sorcerers Bundle (Books 2 and 3)**  
*Author:* Morgan Rice  
*Why it fits:* The bundle follows Kyra’s quest to the mysterious Tower of Ur and features Alec’s “unique skills in the forge” to aid the resistance. The story blends mystery (the secrets of the tower and the cryptic uncle) with magical elements that involve crafting and alchemical‑like forging.

**2. The Weight of Honor (Kings and Sorcerers – Book 3)**  
*Author:* Morgan Rice  
*Why it fits:* This installment deepens the mystery surrounding Kyra’s uncle and the Tower of Ur, while Alec must “tap his unique skills in the forge” to help the resistance. The narrative intertwines investigative intrigue with magical forging, offering both mystery and a craft reminiscent of alchemy.

**3. Night of the Bold (Kings and Sorcerers – Book 6)**  
*Author:* Morgan Rice  
*Why it fits:* The climax involves Kyra’s search for the Staff of Truth and the uncovering of hidden secrets, providing a strong mystery thread. The plot al

In [31]:
query = "educational books"

result = ask_book_rag(query)

print(result["answer"])

**1. The Law of Success in Sixteen Lessons**  
*Author:* Napoleon Hill  
*Why it fits:* This book is a foundational self‑help text that teaches readers the principles that differentiate successful people from the average. It offers clear, actionable lessons on personal growth and success, making it an educational resource for anyone looking to improve their mindset and achieve their goals.

**2. Freakonomics Rev Ed: A Rogue Economist Explores the Hidden Side of Everything**  
*Author:* Steven D. Levitt  
*Why it fits:* Levitt’s book uses economic theory to explain everyday phenomena, from crime rates to parenting choices. It’s a highly readable, thought‑provoking guide that educates readers about incentives, competition, and how economic thinking can illuminate common life questions.


In [32]:
query = "cook books"

result = ask_book_rag(query)

print(result["answer"])

**Recommendation**

- **Title:** 15 Delicious Slow Cooker Recipes  
- **Author:** Sallie Stone  
- **Why it matches:** This book is a dedicated cookbook that offers a variety of slow‑cooker recipes, such as Sweet and Sour Pork, Beef Stew, Turkey Chili, Seafood Delis, and Swiss Steak. It provides detailed ingredient lists, step‑by‑step instructions, and nutritional information, making it a practical resource for anyone looking to explore or expand their cooking repertoire.


In [33]:
query = "stories that occurs on the sea"

result = ask_book_rag(query)

print(result["answer"])

**Recommendation**

- **Title:** *Moby Dick. Illustrated edition*  
- **Author:** Melville Herman  
- **Why it fits:** The novel follows the whaling voyage of the Pequod, with the entire narrative unfolding on the open ocean. It centers on the pursuit of a giant white whale, making the sea the primary setting and driving force of the story.


In [37]:
query = "scary books"

result = ask_book_rag(query)

print(result["answer"])

**Recommendation**

- **Title:** *Salem's Lot*  
- **Author:** Stephen King  
- **Why it fits:** The book is listed under the “Horror” genre and its description centers on vampires, supernatural terror, and escalating dread in a small town. These elements make it a quintessential scary read.


In [38]:
import random
import re
import pandas as pd

In [39]:


def build_eval_queries(books, n=20, seed=42):
    random.seed(seed)


    unique_books = {}
    for book in books:
        title = book["title"].strip()
        if title and title not in unique_books:
            unique_books[title] = book

    candidates = list(unique_books.values())
    selected = random.sample(candidates, min(n, len(candidates)))

    queries = []

    for book in selected:
        title = book["title"]
        genre = book["genre"].strip()

        if genre and genre.lower() != "none":
            query = f"recommend books in the genre {genre}"
        else:

            words = re.findall(r"\b[a-zA-Z]{5,}\b", book["description"])
            keywords = " ".join(words[:5])
            query = f"recommend books related to {keywords}"

        queries.append({
            "query": query,
            "expected_title": title
        })

    return queries


eval_queries = build_eval_queries(books, n=20)

pd.DataFrame(eval_queries).head()

,query,expected_title
0,"recommend books in the genre Fiction, Classics",And Then There Were None
1,recommend books related to SUNDAY TIMES BESTSE...,Tall Tales and Wee Stories: The Best of Billy ...
2,"recommend books in the genre Fiction, Media Ti...",God of War: The Official Novelization
3,"recommend books in the genre Business &amp, Ec...",The Essentials of Finance and Accounting for N...
4,recommend books related to eight Times Today b...,Thirty-Five and a Half Conspiracies: Rose Gard...


In [40]:

def hit_rate_at_k(results, expected_title, k=5):
    expected_title = expected_title.lower().strip()

    for book in results[:k]:
        if book["title"].lower().strip() == expected_title:
            return 1

    return 0


def reciprocal_rank_at_k(results, expected_title, k=5):
    expected_title = expected_title.lower().strip()

    for rank, book in enumerate(results[:k], start=1):
        if book["title"].lower().strip() == expected_title:
            return 1 / rank

    return 0

In [41]:
# ============================================
# COMPARE RETRIEVAL METHODS
# ============================================

def evaluate_retrieval(eval_queries, k=5):

    rows = []

    methods = {
        "keyword": keyword_search,
        "vector": vector_search,
        "hybrid": hybrid_search
    }

    for item in eval_queries:

        query = item["query"]
        expected_title = item["expected_title"]

        for method_name, search_function in methods.items():

            results = search_function(query, k=k)

            rows.append({
                "query": query,
                "expected_title": expected_title,
                "method": method_name,
                "hit_rate": hit_rate_at_k(
                    results,
                    expected_title,
                    k
                ),
                "mrr": reciprocal_rank_at_k(
                    results,
                    expected_title,
                    k
                )
            })

    return pd.DataFrame(rows)

In [42]:
retrieval_results = evaluate_retrieval(eval_queries, k=5)

retrieval_results

,query,expected_title,method,hit_rate,mrr
0,"recommend books in the genre Fiction, Classics",And Then There Were None,keyword,0,0.000000
1,"recommend books in the genre Fiction, Classics",And Then There Were None,vector,0,0.000000
2,"recommend books in the genre Fiction, Classics",And Then There Were None,hybrid,0,0.000000
3,recommend books related to SUNDAY TIMES BESTSE...,Tall Tales and Wee Stories: The Best of Billy ...,keyword,0,0.000000
4,recommend books related to SUNDAY TIMES BESTSE...,Tall Tales and Wee Stories: The Best of Billy ...,vector,0,0.000000
5,recommend books related to SUNDAY TIMES BESTSE...,Tall Tales and Wee Stories: The Best of Billy ...,hybrid,0,0.000000
6,"recommend books in the genre Fiction, Media Ti...",God of War: The Official Novelization,keyword,1,0.333333
7,"recommend books in the genre Fiction, Media Ti...",God of War: The Official Novelization,vector,0,0.000000
8,"recommend books in the genre Fiction, Media Ti...",God of War: The Official Novelization,hybrid,1,0.200000
9,"recommend books in the genre Business &amp, Ec...",The Essentials of Finance and Accounting for N...,keyword,1,0.500000


In [43]:
retrieval_summary = (
    retrieval_results
    .groupby("method")
    .agg(
        hit_rate_at_5=("hit_rate", "mean"),
        mrr_at_5=("mrr", "mean")
    )
    .sort_values("mrr_at_5", ascending=False)
)

retrieval_summary

,hit_rate_at_5,mrr_at_5
method,,
keyword,0.50,0.370833
hybrid,0.45,0.285000
vector,0.15,0.085000
